```
**Data modeling** = deciding how to structure data so it can be:

Stored efficiently

Queried correctly

Understood by humans

Maintained over time
```

## Star Schema

### Structure

**One fact table** in the center, surrounded by **flat (denormalized) dimension tables**.

```text
                [dim_date]
                     |
                     |
[dim_product] — [fact_sales] — [dim_customer]
                     |
                     |
                [dim_store]
```

### The Tables

`fact_sales` (the center — measures + foreign keys)

| sale_id | date_key | product_key | customer_key | store_key | quantity | revenue |
| ------- | -------- | ----------- | ------------ | --------- | -------- | ------- |
| 1       | 20240215 | 101         | 201          | 301       | 2        | 250     |
| 2       | 20240215 | 102         | 202          | 301       | 1        | 150     |
| 3       | 20240216 | 101         | 203          | 302       | 3        | 375     |

`dim_product` (flat — everything in one table)

| product_key | product_name | subcategory | category    | brand |
| ----------- | ------------ | ----------- | ----------- | ----- |
| 101         | Widget Pro   | Widgets     | Electronics | Acme  |
| 102         | Gadget Mini  | Gadgets     | Electronics | Beta  |

`dim_customer` (flat)

| customer_key | name  | city   | state | country | segment  |
| ------------ | ----- | ------ | ----- | ------- | -------- |
| 201          | Alice | NYC    | NY    | USA     | Premium  |
| 202          | Bob   | Boston | MA    | USA     | Standard |
| 203          | Carol | Miami  | FL    | USA     | Premium  |

`dim_store` (flat)

| store_key | store_name    | city  | state | region    |
| --------- | ------------- | ----- | ----- | --------- |
| 301       | Downtown NYC  | NYC   | NY    | Northeast |
| 302       | Miami Central | Miami | FL    | Southeast |

`dim_date`

| date_key | date       | day | month | quarter | year |
| -------- | ---------- | --- | ----- | ------- | ---- |
| 20240215 | 2024-02-15 | 15  | 2     | Q1      | 2024 |

### Key Characteristics

| Feature          | Value                                           |
| ---------------- | ----------------------------------------------- |
| Number of tables | 1 fact + 4 dims = **5 tables**                  |
| Dimension shape  | Flat (denormalized)                             |
| Redundancy       | Higher (category repeated in every product row) |
| Joins needed     | **Few**                                         |
| Query complexity | Simple                                          |
| Query speed      | Fast                                            |

### Typical Query

```sql
SELECT
    p.category,
    d.year,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_product p ON f.product_key = p.product_key
JOIN dim_date d ON f.date_key = d.date_key
WHERE d.year = 2024
GROUP BY p.category, d.year;
```

**Only 2 joins.** Clean and fast.

---

## Snowflake Schema

### Structure

**Same fact table**, but dimensions are **normalized** — split into multiple related tables.

```text
                     [dim_date]
                          |
                          |
[dim_product] — [fact_sales] — [dim_customer]
      |                           |
      v                           v
[dim_subcategory]           [dim_city]
      |                           |
      v                           v
[dim_category]              [dim_state]
                                  |
                                  v
                            [dim_country]
```

### The Tables

`fact_sales` — same as Star

`dim_product` (normalized — only immediate attributes)

| product_key | product_name | subcategory_key | brand |
| ----------- | ------------ | --------------- | ----- |
| 101         | Widget Pro   | 501             | Acme  |
| 102         | Gadget Mini  | 502             | Beta  |

`dim_subcategory`

| subcategory_key | subcategory_name | category_key |
| --------------- | ---------------- | ------------ |
| 501             | Widgets          | 601          |
| 502             | Gadgets          | 601          |

`dim_category`

| category_key | category_name |
| ------------ | ------------- |
| 601          | Electronics   |

`dim_customer` (normalized)

| customer_key | name  | city_key | segment  |
| ------------ | ----- | -------- | -------- |
| 201          | Alice | 701      | Premium  |
| 202          | Bob   | 702      | Standard |
| 203          | Carol | 703      | Premium  |

`dim_city`

| city_key | city_name | state_key |
| -------- | --------- | --------- |
| 701      | NYC       | 801       |
| 702      | Boston    | 802       |
| 703      | Miami     | 803       |

`dim_state`

| state_key | state_name | country_key |
| --------- | ---------- | ----------- |
| 801       | NY         | 901         |
| 802       | MA         | 901         |
| 803       | FL         | 901         |

`dim_country`

| country_key | country_name |
| ----------- | ------------ |
| 901         | USA          |

### Key Characteristics

| Feature          | Value                             |
| ---------------- | --------------------------------- |
| Number of tables | 1 fact + ~10 dims = **11 tables** |
| Dimension shape  | Normalized (splits)               |
| Redundancy       | Lower (each value stored once)    |
| Joins needed     | **Many**                          |
| Query complexity | Complex                           |
| Query speed      | Slower (more joins)               |

### Typical Query

```sql
SELECT
    cat.category_name,
    d.year,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_product p
    ON f.product_key = p.product_key
JOIN dim_subcategory sub
    ON p.subcategory_key = sub.subcategory_key
JOIN dim_category cat
    ON sub.category_key = cat.category_key
JOIN dim_date d
    ON f.date_key = d.date_key
WHERE d.year = 2024
GROUP BY cat.category_name, d.year;
```
